In [5]:
import sys
!{sys.executable} -m pip install pydeseq2

  Using cached formulaic_contrasts-1.0.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached session_info-1.0.1-py3-none-any.whl.metadata (5.1 kB)
  Using cached stdlib_list-0.12.0-py3-none-any.whl.metadata (3.3 kB)
Using cached formulaic_contrasts-1.0.0-py3-none-any.whl (10 kB)
Using cached session_info-1.0.1-py3-none-any.whl (9.1 kB)
Using cached stdlib_list-0.12.0-py3-none-any.whl (87 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pydeseq2]


In [1]:
import os
import pandas as pd
import numpy as np

# Absolute path to mouse raw count matrix
counts_path = os.path.expanduser("~/FYP_Regeneration/data/mouse/processed/GSE131078_gene_counts.txt.gz")
print(f"Loading mouse bulk count matrix from:\n{counts_path}")

# Load dataset
counts_df = pd.read_csv(counts_path, sep="\t", index_col=0)

# Drop gene metadata columns if present (e.g., 'length')
if 'length' in counts_df.columns:
    counts_df = counts_df.drop(columns=['length'])

print(f"Cleaned Matrix shape: {counts_df.shape[0]} genes x {counts_df.shape[1]} samples\n")
print("All Sample Names in Matrix:")
for i, col in enumerate(counts_df.columns, 1):
    print(f"{i:02d}. {col}")

Loading mouse bulk count matrix from:
/home/ghayyas/FYP_Regeneration/data/mouse/processed/GSE131078_gene_counts.txt.gz
Cleaned Matrix shape: 15055 genes x 24 samples

All Sample Names in Matrix:
01. R12_1
02. R12_2
03. R12_3
04. R12_4
05. R14_1
06. R14_2
07. R14_3
08. R14_4
09. R21_1
10. R21_2
11. R21_3
12. R21_4
13. N12_1
14. N12_2
15. N12_3
16. N12_4
17. N14_1
18. N14_2
19. N14_3
20. N14_4
21. N21_1
22. N21_2
23. N21_3
24. N21_4


In [2]:
# Cell 2: Metadata Design Matrix, PyDESeq2 Execution, and Export
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

# 1. Parse Metadata Design Matrix
sample_names = counts_df.columns.tolist()
metadata = []

for sample in sample_names:
    condition = "Regenerative" if sample.startswith("R") else "Non_Regenerative"
    if "12" in sample:
        timepoint = "D12"
    elif "14" in sample:
        timepoint = "D14"
    elif "21" in sample:
        timepoint = "D21"
    else:
        timepoint = "Unknown"
        
    metadata.append({"sample": sample, "condition": condition, "timepoint": timepoint})

meta_df = pd.DataFrame(metadata).set_index("sample")
print("Parsed Experimental Design:")
print(pd.crosstab(meta_df['condition'], meta_df['timepoint']))

# 2. Prepare Counts Matrix for PyDESeq2 (Samples as rows, Genes as columns, integer values)
counts_transposed = counts_df.T.astype(int)

# Filter low-expression genes (keep genes with at least 10 total counts across samples)
gene_mask = counts_transposed.sum(axis=0) >= 10
counts_filtered = counts_transposed.loc[:, gene_mask]
print(f"\nFiltered gene count matrix: {counts_filtered.shape[1]} genes retained (out of {counts_transposed.shape[1]}).")

# 3. Initialize & Run PyDESeq2
print("\nRunning PyDESeq2 pipeline (Regenerative vs Non_Regenerative)...")
dds = DeseqDataSet(
    counts=counts_filtered,
    metadata=meta_df,
    design_factors="condition",
    refit_cooks=True,
    n_cpus=4
)
dds.deseq2()

# 4. Extract Differential Expression Statistics
stat_res = DeseqStats(dds, contrast=["condition", "Regenerative", "Non_Regenerative"], n_cpus=4)
stat_res.summary()

res_df = stat_res.results_df.copy()
res_df['gene'] = res_df.index

# Filter significant genes (padj < 0.05 and |log2FoldChange| > 1.0)
sig_df = res_df[(res_df['padj'] < 0.05) & (res_df['log2FoldChange'].abs() > 1.0)].copy()

# 5. Export Output Tables
tables_dir = os.path.expanduser("~/FYP_Regeneration/results/tables/")
figures_dir = os.path.expanduser("~/FYP_Regeneration/results/figures/mouse/")
os.makedirs(tables_dir, exist_ok=True)
os.makedirs(figures_dir, exist_ok=True)

res_df.to_csv(os.path.join(tables_dir, "mouse_de_genes_all.csv"), index=False)
sig_df.to_csv(os.path.join(tables_dir, "mouse_de_genes_sig.csv"), index=False)

# 6. Generate & Save Volcano Plot
plt.figure(figsize=(8, 6))
res_df['-log10_padj'] = -np.log10(res_df['padj'].fillna(1))

# Color points by significance
plt.scatter(res_df['log2FoldChange'], res_df['-log10_padj'], c='grey', alpha=0.5, s=10, label='Not Sig')
plt.scatter(sig_df['log2FoldChange'], -np.log10(sig_df['padj']), c='crimson', alpha=0.7, s=12, label='Significant')

plt.axvline(x=1.0, color='black', linestyle='--', linewidth=0.8)
plt.axvline(x=-1.0, color='black', linestyle='--', linewidth=0.8)
plt.axhline(y=-np.log10(0.05), color='black', linestyle='--', linewidth=0.8)

plt.xlabel("log2 Fold Change (Regenerative / Non-Regenerative)")
plt.ylabel("-log10 Adjusted P-value")
plt.title("Mouse Digit-Tip Bulk RNA-seq Differential Expression")
plt.legend(loc='upper right')
plt.tight_layout()

volcano_path = os.path.join(figures_dir, "volcano_mouse_de.png")
plt.savefig(volcano_path, dpi=150)
plt.close()

print("\nStep 3 Execution Complete!")
print(f"- Total DE results saved to: {tables_dir}mouse_de_genes_all.csv")
print(f"- Significant DE genes saved to: {tables_dir}mouse_de_genes_sig.csv")
print(f"- Volcano plot saved to: {volcano_path}")
print(f"- Total significant DE genes identified: {len(sig_df)}")

Parsed Experimental Design:
timepoint         D12  D14  D21
condition                      
Non_Regenerative    4    4    4
Regenerative        4    4    4

Filtered gene count matrix: 15055 genes retained (out of 15055).

Running PyDESeq2 pipeline (Regenerative vs Non_Regenerative)...
Using None as control genes, passed at DeseqDataSet initialization


/tmp/ipykernel_4381/1197718345.py:41: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(
Fitting size factors...
... done in 0.02 seconds.

Fitting dispersions...
... done in 5.64 seconds.

Fitting dispersion trend curve...
... done in 0.48 seconds.

Fitting MAP dispersions...
... done in 8.88 seconds.

Fitting LFCs...
... done in 4.51 seconds.

Calculating cook's distance...
... done in 0.11 seconds.

Replacing 19 outlier genes.

Fitting dispersions...
... done in 0.02 seconds.

Fitting MAP dispersions...
... done in 0.02 seconds.

Fitting LFCs...
... done in 0.02 seconds.

Running Wald tests...
... done in 2.08 seconds.



Log2 fold change & Wald test p-value: condition Regenerative vs Non_Regenerative
                  baseMean  log2FoldChange     lfcSE      stat    pvalue  \
name                                                                       
0610009B22Rik   161.686626        0.091912  0.097076  0.946801  0.343740   
0610009L18Rik    22.512360       -0.061164  0.152542 -0.400962  0.688448   
0610009O20Rik   310.797709        0.030704  0.077700  0.395158  0.692726   
0610010F05Rik   426.674312       -0.182045  0.083417 -2.182338  0.029085   
0610012G03Rik   457.251543        0.055773  0.071562  0.779362  0.435766   
...                    ...             ...       ...       ...       ...   
Zxdc            937.586471       -0.043697  0.074309 -0.588048  0.556500   
Zyg11b         1982.829741        0.041112  0.042218  0.973807  0.330153   
Zyx            1502.978552        0.140241  0.103842  1.350515  0.176851   
Zzef1          2200.025895       -0.069916  0.037261 -1.876383  0.060603   
Zzz3   